# Module 6.2: Monitoring & Alerting Setup

## Learning Objectives
By completing this notebook, you will:
1. Build monitoring dashboards for GenAI commodity pipelines
2. Implement alerting for failures and anomalies
3. Track LLM performance and quality metrics
4. Monitor signal quality drift over time

## Prerequisites
- Completion of Module 6.1 (Pipeline Build)
- Understanding of monitoring best practices
- Familiarity with observability concepts

## Estimated Time: 90 minutes

---

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
learning_objectives([
    "Completion of Module 6.1 (Pipeline Build)",
    "Understanding of monitoring best practices",
    "Familiarity with observability concepts"
])

## Setup & Imports

In [ ]:
# Purpose: Import libraries for monitoring
# Key Concept: Observability and alerting infrastructure

import os
import json
import time
import logging
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Any, Callable
from dataclasses import dataclass, field, asdict
from enum import Enum
from collections import deque, defaultdict
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
np.random.seed(42)

print("✓ Imports complete")

## 1. Understanding Production Monitoring

### Why Monitor GenAI Pipelines?

GenAI pipelines have unique monitoring requirements:

**Traditional Metrics:**
- Latency (pipeline execution time)
- Error rate (% of failed runs)
- Data freshness (time since last update)
- Resource usage (API calls, costs)

**LLM-Specific Metrics:**
- Confidence score distribution
- Signal quality drift
- Extraction accuracy
- Cost per run

### Monitoring Layers

```
┌─────────────────────────────────────┐
│  INFRASTRUCTURE MONITORING          │
│  (CPU, memory, network)             │
└─────────────────┬───────────────────┘
                  │
┌─────────────────▼───────────────────┐
│  APPLICATION MONITORING             │
│  (pipeline runs, errors, latency)   │
└─────────────────┬───────────────────┘
                  │
┌─────────────────▼───────────────────┐
│  DATA QUALITY MONITORING            │
│  (freshness, completeness)          │
└─────────────────┬───────────────────┘
                  │
┌─────────────────▼───────────────────┐
│  MODEL MONITORING                   │
│  (LLM performance, signal quality)  │
└─────────────────────────────────────┘
```

### Alerting Strategy

1. **Critical**: Pipeline complete failure, data source down
2. **Warning**: High latency, partial failures, quality degradation
3. **Info**: Successful runs, scheduled maintenance

## 2. Metrics Collection System

In [ ]:
# Purpose: Build metrics collection infrastructure
# Key Concept: Time-series metrics storage and aggregation

class MetricType(str, Enum):
    """Types of metrics."""
    COUNTER = "counter"  # Incremental count
    GAUGE = "gauge"      # Point-in-time value
    HISTOGRAM = "histogram"  # Distribution
    TIMER = "timer"      # Duration measurement


@dataclass
class MetricPoint:
    """Single metric data point."""
    
    name: str
    value: float
    timestamp: datetime
    metric_type: MetricType
    tags: Dict[str, str] = field(default_factory=dict)


class MetricsCollector:
    """Collect and store metrics."""
    
    def __init__(self, retention_hours: int = 168):  # 7 days default
        self.metrics: Dict[str, List[MetricPoint]] = defaultdict(list)
        self.retention_hours = retention_hours
        self.logger = logging.getLogger(f"{__name__}.MetricsCollector")
    
    def record(self, 
               name: str,
               value: float,
               metric_type: MetricType = MetricType.GAUGE,
               tags: Optional[Dict[str, str]] = None):
        """
        Record a metric point.
        
        Parameters
        ----------
        name : str
            Metric name
        value : float
            Metric value
        metric_type : MetricType
            Type of metric
        tags : dict, optional
            Additional tags/dimensions
        """
        point = MetricPoint(
            name=name,
            value=value,
            timestamp=datetime.now(),
            metric_type=metric_type,
            tags=tags or {}
        )
        
        self.metrics[name].append(point)
        self._cleanup_old_metrics()
    
    def increment(self, name: str, tags: Optional[Dict[str, str]] = None):
        """Increment a counter."""
        self.record(name, 1, MetricType.COUNTER, tags)
    
    def timing(self, name: str, duration: float, tags: Optional[Dict[str, str]] = None):
        """Record a timing/duration."""
        self.record(name, duration, MetricType.TIMER, tags)
    
    def get_metric(self, 
                   name: str,
                   start_time: Optional[datetime] = None,
                   end_time: Optional[datetime] = None) -> pd.DataFrame:
        """
        Get metric time series.
        
        Parameters
        ----------
        name : str
            Metric name
        start_time : datetime, optional
            Start of time range
        end_time : datetime, optional
            End of time range
            
        Returns
        -------
        pd.DataFrame
            Metric time series
        """
        if name not in self.metrics:
            return pd.DataFrame()
        
        points = self.metrics[name]
        
        # Filter by time range
        if start_time:
            points = [p for p in points if p.timestamp >= start_time]
        if end_time:
            points = [p for p in points if p.timestamp <= end_time]
        
        # Convert to DataFrame
        data = [{
            'timestamp': p.timestamp,
            'value': p.value,
            **p.tags
        } for p in points]
        
        return pd.DataFrame(data)
    
    def aggregate(self, 
                 name: str,
                 agg_func: str = 'mean',
                 window: str = '1H') -> pd.DataFrame:
        """
        Aggregate metric over time windows.
        
        Parameters
        ----------
        name : str
            Metric name
        agg_func : str
            Aggregation function (mean, sum, min, max, count)
        window : str
            Time window (pandas frequency string)
            
        Returns
        -------
        pd.DataFrame
            Aggregated time series
        """
        df = self.get_metric(name)
        if df.empty:
            return df
        
        df = df.set_index('timestamp')
        aggregated = df['value'].resample(window).agg(agg_func)
        
        return aggregated.reset_index()
    
    def _cleanup_old_metrics(self):
        """Remove metrics older than retention period."""
        cutoff = datetime.now() - timedelta(hours=self.retention_hours)
        
        for name in self.metrics:
            self.metrics[name] = [
                p for p in self.metrics[name] if p.timestamp > cutoff
            ]
    
    def list_metrics(self) -> List[str]:
        """List all available metrics."""
        return sorted(self.metrics.keys())
    
    def stats(self) -> Dict:
        """Get collector statistics."""
        total_points = sum(len(points) for points in self.metrics.values())
        
        return {
            'total_metrics': len(self.metrics),
            'total_points': total_points,
            'retention_hours': self.retention_hours
        }


# Initialize metrics collector
metrics = MetricsCollector()

# Record some sample metrics
for i in range(10):
    metrics.record('pipeline.latency', np.random.uniform(1, 5), MetricType.TIMER)
    metrics.increment('pipeline.runs', tags={'status': 'success'})
    time.sleep(0.01)

print(f"✓ Metrics collector initialized")
print(f"  Stats: {metrics.stats()}")
print(f"  Metrics: {metrics.list_metrics()}")

## 3. Alerting System

In [ ]:
# Purpose: Build alerting infrastructure
# Key Concept: Rule-based alerts with notification routing

class AlertSeverity(str, Enum):
    """Alert severity levels."""
    CRITICAL = "critical"
    WARNING = "warning"
    INFO = "info"


@dataclass
class Alert:
    """Alert notification."""
    
    name: str
    severity: AlertSeverity
    message: str
    timestamp: datetime
    metric_name: Optional[str] = None
    metric_value: Optional[float] = None
    threshold: Optional[float] = None
    
    def to_dict(self) -> Dict:
        return asdict(self)


@dataclass
class AlertRule:
    """Alert rule definition."""
    
    name: str
    metric_name: str
    condition: Callable[[float], bool]  # Function that returns True if alert should fire
    severity: AlertSeverity
    message_template: str
    cooldown_minutes: int = 60  # Minimum time between alerts
    last_fired: Optional[datetime] = None


class AlertManager:
    """Manage alerts and notifications."""
    
    def __init__(self, metrics_collector: MetricsCollector):
        self.metrics = metrics_collector
        self.rules: List[AlertRule] = []
        self.alerts: List[Alert] = []
        self.logger = logging.getLogger(f"{__name__}.AlertManager")
    
    def add_rule(self, rule: AlertRule):
        """Add alert rule."""
        self.rules.append(rule)
        self.logger.info(f"Added alert rule: {rule.name}")
    
    def check_alerts(self):
        """
        Check all alert rules and fire alerts as needed.
        """
        for rule in self.rules:
            # Check cooldown
            if rule.last_fired:
                time_since_last = (datetime.now() - rule.last_fired).total_seconds() / 60
                if time_since_last < rule.cooldown_minutes:
                    continue
            
            # Get latest metric value
            metric_df = self.metrics.get_metric(rule.metric_name)
            if metric_df.empty:
                continue
            
            latest_value = metric_df.iloc[-1]['value']
            
            # Check condition
            if rule.condition(latest_value):
                # Fire alert
                alert = Alert(
                    name=rule.name,
                    severity=rule.severity,
                    message=rule.message_template.format(value=latest_value),
                    timestamp=datetime.now(),
                    metric_name=rule.metric_name,
                    metric_value=latest_value
                )
                
                self._fire_alert(alert)
                rule.last_fired = datetime.now()
    
    def _fire_alert(self, alert: Alert):
        """
        Fire an alert (send notification).
        
        In production, this would:
        - Send email
        - Post to Slack/Teams
        - Create PagerDuty incident
        - Log to monitoring system
        """
        self.alerts.append(alert)
        
        # Log alert
        log_level = {
            AlertSeverity.CRITICAL: logging.CRITICAL,
            AlertSeverity.WARNING: logging.WARNING,
            AlertSeverity.INFO: logging.INFO
        }[alert.severity]
        
        self.logger.log(
            log_level,
            f"ALERT [{alert.severity.value.upper()}] {alert.name}: {alert.message}"
        )
        
        # In production: send notifications
        # self._send_email(alert)
        # self._post_to_slack(alert)
    
    def get_active_alerts(self, 
                         severity: Optional[AlertSeverity] = None) -> List[Alert]:
        """Get recent alerts."""
        alerts = self.alerts
        
        if severity:
            alerts = [a for a in alerts if a.severity == severity]
        
        # Return alerts from last 24 hours
        cutoff = datetime.now() - timedelta(hours=24)
        return [a for a in alerts if a.timestamp > cutoff]
    
    def summary(self) -> pd.DataFrame:
        """Get alert summary."""
        if not self.alerts:
            return pd.DataFrame()
        
        data = [a.to_dict() for a in self.alerts]
        df = pd.DataFrame(data)
        
        summary = df.groupby('severity').agg({
            'name': 'count'
        }).rename(columns={'name': 'count'})
        
        return summary


# Initialize alert manager
alert_manager = AlertManager(metrics)

# Define alert rules
alert_manager.add_rule(AlertRule(
    name="High Pipeline Latency",
    metric_name="pipeline.latency",
    condition=lambda x: x > 4.0,  # Alert if latency > 4 seconds
    severity=AlertSeverity.WARNING,
    message_template="Pipeline latency is high: {value:.2f}s",
    cooldown_minutes=30
))

alert_manager.add_rule(AlertRule(
    name="Pipeline Failure",
    metric_name="pipeline.errors",
    condition=lambda x: x > 0,
    severity=AlertSeverity.CRITICAL,
    message_template="Pipeline failed: {value:.0f} errors",
    cooldown_minutes=15
))

# Simulate high latency
metrics.record('pipeline.latency', 5.2, MetricType.TIMER)

# Check alerts
alert_manager.check_alerts()

print("\n=== ACTIVE ALERTS ===")
for alert in alert_manager.get_active_alerts():
    print(f"[{alert.severity.value.upper()}] {alert.message}")

## 4. LLM Performance Monitoring

In [ ]:
# Purpose: Monitor LLM-specific metrics
# Key Concept: Tracking model performance and quality drift

@dataclass
class LLMMetrics:
    """LLM performance metrics for a single run."""
    
    timestamp: datetime
    operation: str  # 'extraction', 'sentiment', 'signal'
    latency_ms: float
    tokens_input: int
    tokens_output: int
    cost_usd: float
    confidence_score: Optional[float] = None
    success: bool = True
    error: Optional[str] = None


class LLMMonitor:
    """Monitor LLM performance and quality."""
    
    def __init__(self, metrics_collector: MetricsCollector):
        self.metrics = metrics_collector
        self.llm_metrics: List[LLMMetrics] = []
        self.logger = logging.getLogger(f"{__name__}.LLMMonitor")
    
    def record_llm_call(self, llm_metrics: LLMMetrics):
        """
        Record metrics from an LLM call.
        
        Parameters
        ----------
        llm_metrics : LLMMetrics
            LLM call metrics
        """
        self.llm_metrics.append(llm_metrics)
        
        # Record to general metrics collector
        self.metrics.timing(
            'llm.latency',
            llm_metrics.latency_ms,
            tags={'operation': llm_metrics.operation}
        )
        
        self.metrics.record(
            'llm.tokens.input',
            llm_metrics.tokens_input,
            tags={'operation': llm_metrics.operation}
        )
        
        self.metrics.record(
            'llm.tokens.output',
            llm_metrics.tokens_output,
            tags={'operation': llm_metrics.operation}
        )
        
        self.metrics.record(
            'llm.cost',
            llm_metrics.cost_usd,
            tags={'operation': llm_metrics.operation}
        )
        
        if llm_metrics.confidence_score is not None:
            self.metrics.record(
                'llm.confidence',
                llm_metrics.confidence_score,
                tags={'operation': llm_metrics.operation}
            )
        
        if not llm_metrics.success:
            self.metrics.increment(
                'llm.errors',
                tags={'operation': llm_metrics.operation}
            )
    
    def get_cost_summary(self, hours: int = 24) -> Dict:
        """
        Get cost summary for recent period.
        
        Parameters
        ----------
        hours : int
            Lookback period in hours
            
        Returns
        -------
        dict
            Cost summary
        """
        cutoff = datetime.now() - timedelta(hours=hours)
        recent = [m for m in self.llm_metrics if m.timestamp > cutoff]
        
        if not recent:
            return {'total_cost': 0, 'total_calls': 0}
        
        total_cost = sum(m.cost_usd for m in recent)
        total_tokens_in = sum(m.tokens_input for m in recent)
        total_tokens_out = sum(m.tokens_output for m in recent)
        
        by_operation = defaultdict(lambda: {'count': 0, 'cost': 0})
        for m in recent:
            by_operation[m.operation]['count'] += 1
            by_operation[m.operation]['cost'] += m.cost_usd
        
        return {
            'period_hours': hours,
            'total_calls': len(recent),
            'total_cost': total_cost,
            'total_tokens_input': total_tokens_in,
            'total_tokens_output': total_tokens_out,
            'avg_cost_per_call': total_cost / len(recent) if recent else 0,
            'by_operation': dict(by_operation)
        }
    
    def detect_confidence_drift(self, window_size: int = 50) -> Dict:
        """
        Detect drift in LLM confidence scores.
        
        Parameters
        ----------
        window_size : int
            Size of sliding window for comparison
            
        Returns
        -------
        dict
            Drift analysis
        """
        # Get recent confidence scores
        with_confidence = [m for m in self.llm_metrics if m.confidence_score is not None]
        
        if len(with_confidence) < window_size * 2:
            return {'drift_detected': False, 'reason': 'Insufficient data'}
        
        # Compare recent window to baseline window
        recent_window = [m.confidence_score for m in with_confidence[-window_size:]]
        baseline_window = [m.confidence_score for m in with_confidence[-2*window_size:-window_size]]
        
        recent_mean = np.mean(recent_window)
        baseline_mean = np.mean(baseline_window)
        
        # Calculate drift
        drift_pct = ((recent_mean - baseline_mean) / baseline_mean) * 100
        
        # Detect significant drift (>10% change)
        drift_detected = abs(drift_pct) > 10
        
        return {
            'drift_detected': drift_detected,
            'recent_mean': recent_mean,
            'baseline_mean': baseline_mean,
            'drift_pct': drift_pct,
            'window_size': window_size
        }


# Initialize LLM monitor
llm_monitor = LLMMonitor(metrics)

# Simulate LLM calls
for i in range(100):
    operation = np.random.choice(['extraction', 'sentiment', 'signal'])
    
    llm_monitor.record_llm_call(LLMMetrics(
        timestamp=datetime.now() - timedelta(minutes=100-i),
        operation=operation,
        latency_ms=np.random.uniform(200, 1500),
        tokens_input=np.random.randint(500, 2000),
        tokens_output=np.random.randint(100, 500),
        cost_usd=np.random.uniform(0.01, 0.08),
        confidence_score=np.random.beta(5, 2) if operation == 'signal' else None,
        success=np.random.random() > 0.05
    ))

# Get cost summary
cost_summary = llm_monitor.get_cost_summary(hours=24)

print("=== LLM COST SUMMARY (24h) ===")
print(f"Total Calls: {cost_summary['total_calls']}")
print(f"Total Cost: ${cost_summary['total_cost']:.2f}")
print(f"Avg Cost/Call: ${cost_summary['avg_cost_per_call']:.4f}")
print(f"Total Tokens (in/out): {cost_summary['total_tokens_input']:,} / {cost_summary['total_tokens_output']:,}")

print("\nBy Operation:")
for op, stats in cost_summary['by_operation'].items():
    print(f"  {op}: {stats['count']} calls, ${stats['cost']:.2f}")

# Check for drift
drift_analysis = llm_monitor.detect_confidence_drift()
print("\n=== CONFIDENCE DRIFT ANALYSIS ===")
print(f"Drift Detected: {drift_analysis['drift_detected']}")
if drift_analysis['drift_detected']:
    print(f"  Recent Mean: {drift_analysis['recent_mean']:.2%}")
    print(f"  Baseline Mean: {drift_analysis['baseline_mean']:.2%}")
    print(f"  Drift: {drift_analysis['drift_pct']:+.1f}%")

## 5. Monitoring Dashboard

In [ ]:
# Purpose: Create monitoring dashboard visualization
# Key Concept: Real-time operational visibility

def create_monitoring_dashboard(metrics_collector: MetricsCollector,
                               llm_monitor: LLMMonitor):
    """
    Create comprehensive monitoring dashboard.
    
    Parameters
    ----------
    metrics_collector : MetricsCollector
        Metrics collector instance
    llm_monitor : LLMMonitor
        LLM monitor instance
    """
    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
    # Plot 1: Pipeline Latency Over Time
    ax1 = fig.add_subplot(gs[0, :])
    latency_df = metrics_collector.get_metric('pipeline.latency')
    if not latency_df.empty:
        ax1.plot(latency_df['timestamp'], latency_df['value'], 
                linewidth=2, marker='o', markersize=4, color='blue')
        ax1.axhline(y=4.0, color='red', linestyle='--', linewidth=1, 
                   label='Alert Threshold (4s)')
        ax1.fill_between(latency_df['timestamp'], 0, latency_df['value'],
                        where=latency_df['value']>=4.0, alpha=0.3, color='red')
        ax1.set_ylabel('Latency (seconds)', fontsize=12)
        ax1.set_title('Pipeline Latency', fontsize=14, fontweight='bold')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
    
    # Plot 2: LLM Cost Over Time
    ax2 = fig.add_subplot(gs[1, 0])
    llm_df = pd.DataFrame([asdict(m) for m in llm_monitor.llm_metrics])
    if not llm_df.empty:
        cost_hourly = llm_df.set_index('timestamp')['cost_usd'].resample('1H').sum()
        ax2.bar(cost_hourly.index, cost_hourly.values, 
               width=0.03, color='green', alpha=0.7)
        ax2.set_ylabel('Cost ($)', fontsize=12)
        ax2.set_title('LLM Cost (Hourly)', fontsize=12, fontweight='bold')
        ax2.grid(True, alpha=0.3, axis='y')
    
    # Plot 3: LLM Latency Distribution
    ax3 = fig.add_subplot(gs[1, 1])
    if not llm_df.empty:
        ax3.hist(llm_df['latency_ms'], bins=30, color='purple', alpha=0.7, edgecolor='black')
        ax3.axvline(x=llm_df['latency_ms'].median(), color='red', 
                   linestyle='--', linewidth=2, label=f"Median: {llm_df['latency_ms'].median():.0f}ms")
        ax3.set_xlabel('Latency (ms)', fontsize=12)
        ax3.set_ylabel('Count', fontsize=12)
        ax3.set_title('LLM Latency Distribution', fontsize=12, fontweight='bold')
        ax3.legend()
        ax3.grid(True, alpha=0.3, axis='y')
    
    # Plot 4: Confidence Score Trend
    ax4 = fig.add_subplot(gs[1, 2])
    confidence_df = llm_df[llm_df['confidence_score'].notna()]
    if not confidence_df.empty:
        # Rolling average
        window = 10
        confidence_sorted = confidence_df.sort_values('timestamp')
        rolling_avg = confidence_sorted['confidence_score'].rolling(window=window).mean()
        
        ax4.scatter(range(len(confidence_sorted)), 
                   confidence_sorted['confidence_score'],
                   alpha=0.3, s=20, color='gray', label='Individual Scores')
        ax4.plot(range(len(rolling_avg)), rolling_avg,
                linewidth=2, color='blue', label=f'{window}-Call Rolling Avg')
        ax4.axhline(y=0.6, color='orange', linestyle='--', 
                   linewidth=1, label='Target (60%)')
        ax4.set_xlabel('Call Number', fontsize=12)
        ax4.set_ylabel('Confidence Score', fontsize=12)
        ax4.set_title('Signal Confidence Over Time', fontsize=12, fontweight='bold')
        ax4.legend()
        ax4.grid(True, alpha=0.3)
    
    # Plot 5: Cost by Operation
    ax5 = fig.add_subplot(gs[2, 0])
    if not llm_df.empty:
        cost_by_op = llm_df.groupby('operation')['cost_usd'].sum()
        ax5.pie(cost_by_op.values, labels=cost_by_op.index, autopct='%1.1f%%',
               startangle=90)
        ax5.set_title('Cost Distribution by Operation', fontsize=12, fontweight='bold')
    
    # Plot 6: Success Rate
    ax6 = fig.add_subplot(gs[2, 1])
    if not llm_df.empty:
        success_rate = llm_df.groupby('operation')['success'].mean() * 100
        colors = ['green' if x >= 95 else 'orange' if x >= 90 else 'red' 
                 for x in success_rate.values]
        bars = ax6.bar(success_rate.index, success_rate.values, 
                      color=colors, alpha=0.7)
        ax6.axhline(y=95, color='red', linestyle='--', linewidth=1, 
                   label='Target (95%)')
        ax6.set_ylabel('Success Rate (%)', fontsize=12)
        ax6.set_title('LLM Success Rate by Operation', fontsize=12, fontweight='bold')
        ax6.legend()
        ax6.grid(True, alpha=0.3, axis='y')
    
    # Plot 7: System Health Summary
    ax7 = fig.add_subplot(gs[2, 2])
    ax7.axis('off')
    
    # Calculate summary stats
    cost_summary = llm_monitor.get_cost_summary(hours=24)
    
    summary_text = f"""
SYSTEM HEALTH SUMMARY
{'='*40}

PIPELINE:
  Active Metrics: {len(metrics_collector.list_metrics())}
  Total Data Points: {metrics_collector.stats()['total_points']}

LLM (24h):
  Total Calls: {cost_summary['total_calls']}
  Total Cost: ${cost_summary['total_cost']:.2f}
  Avg Latency: {llm_df['latency_ms'].mean():.0f}ms
  Success Rate: {(llm_df['success'].mean()*100):.1f}%

ALERTS:
  Critical: {len([a for a in alert_manager.get_active_alerts() if a.severity == AlertSeverity.CRITICAL])}
  Warning: {len([a for a in alert_manager.get_active_alerts() if a.severity == AlertSeverity.WARNING])}
  Info: {len([a for a in alert_manager.get_active_alerts() if a.severity == AlertSeverity.INFO])}
    """
    
    ax7.text(0.1, 0.9, summary_text, transform=ax7.transAxes,
            fontsize=10, verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))
    
    plt.suptitle('Production Monitoring Dashboard', 
                fontsize=16, fontweight='bold', y=0.995)
    plt.show()


# Create dashboard
create_monitoring_dashboard(metrics, llm_monitor)

## 6. Health Check Endpoint

In [ ]:
# Purpose: Create health check for load balancers/orchestrators
# Key Concept: Programmatic health status

class HealthStatus(str, Enum):
    """Health status levels."""
    HEALTHY = "healthy"
    DEGRADED = "degraded"
    UNHEALTHY = "unhealthy"


@dataclass
class HealthCheck:
    """Health check result."""
    
    status: HealthStatus
    timestamp: datetime
    checks: Dict[str, bool]
    metrics: Dict[str, Any]
    message: Optional[str] = None


class HealthChecker:
    """Perform system health checks."""
    
    def __init__(self, 
                 metrics_collector: MetricsCollector,
                 llm_monitor: LLMMonitor):
        self.metrics = metrics_collector
        self.llm_monitor = llm_monitor
        self.logger = logging.getLogger(f"{__name__}.HealthChecker")
    
    def check_health(self) -> HealthCheck:
        """
        Perform comprehensive health check.
        
        Returns
        -------
        HealthCheck
            Health check result
        """
        checks = {}
        
        # Check 1: Recent pipeline runs
        latency_df = self.metrics.get_metric('pipeline.latency')
        if not latency_df.empty:
            last_run = latency_df.iloc[-1]['timestamp']
            time_since_run = (datetime.now() - last_run).total_seconds() / 60
            checks['recent_pipeline_run'] = time_since_run < 120  # Within 2 hours
        else:
            checks['recent_pipeline_run'] = False
        
        # Check 2: LLM error rate
        cost_summary = self.llm_monitor.get_cost_summary(hours=1)
        if cost_summary['total_calls'] > 0:
            llm_df = pd.DataFrame([asdict(m) for m in self.llm_monitor.llm_metrics])
            recent_hour = datetime.now() - timedelta(hours=1)
            recent_df = llm_df[llm_df['timestamp'] > recent_hour]
            error_rate = 1 - recent_df['success'].mean() if not recent_df.empty else 0
            checks['llm_error_rate_ok'] = error_rate < 0.05  # <5% errors
        else:
            checks['llm_error_rate_ok'] = True
        
        # Check 3: Data freshness
        checks['data_fresh'] = True  # Would check actual data source timestamps
        
        # Determine overall status
        failed_checks = [k for k, v in checks.items() if not v]
        
        if not failed_checks:
            status = HealthStatus.HEALTHY
            message = "All systems operational"
        elif len(failed_checks) == 1:
            status = HealthStatus.DEGRADED
            message = f"Degraded: {failed_checks[0]} failing"
        else:
            status = HealthStatus.UNHEALTHY
            message = f"Unhealthy: {len(failed_checks)} checks failing"
        
        # Collect metrics
        health_metrics = {
            'metrics_count': len(self.metrics.list_metrics()),
            'llm_calls_1h': cost_summary['total_calls'],
            'llm_cost_1h': cost_summary['total_cost']
        }
        
        return HealthCheck(
            status=status,
            timestamp=datetime.now(),
            checks=checks,
            metrics=health_metrics,
            message=message
        )


# Perform health check
health_checker = HealthChecker(metrics, llm_monitor)
health = health_checker.check_health()

print("=== HEALTH CHECK ===")
print(f"Status: {health.status.value.upper()}")
print(f"Message: {health.message}")
print(f"\nChecks:")
for check, passing in health.checks.items():
    symbol = "✓" if passing else "✗"
    print(f"  {symbol} {check}")
print(f"\nMetrics:")
for metric, value in health.metrics.items():
    print(f"  {metric}: {value}")

## Exercise 6.2: Build Custom Monitoring

In [ ]:
# Exercise: Create custom monitoring for signal quality
#
# Task:
# 1. Create a SignalQualityMonitor class
# 2. Track signal win rate over time
# 3. Detect when win rate drops below threshold
# 4. Create alert rule for quality degradation

# YOUR CODE HERE
# ---------------

# Step 1: Create monitor
class SignalQualityMonitor:
    pass  # Implement this

# Step 2: Track win rate
# ...

# Step 3: Detect degradation
# ...

# Step 4: Create alert
# alert_manager.add_rule(...)

## Summary

### Key Takeaways

1. **Comprehensive Monitoring**: Track infrastructure, application, data, and model metrics
2. **LLM-Specific Metrics**: Monitor costs, latency, confidence scores, and quality drift
3. **Proactive Alerting**: Set up rules to catch issues before they impact users
4. **Health Checks**: Provide programmatic health status for orchestration
5. **Dashboards**: Visualize key metrics for operational awareness

### Module 6 Complete

You've now built:
- **6.1**: Production pipeline with error handling, caching, and orchestration
- **6.2**: Comprehensive monitoring and alerting system

### Course Complete!

You've completed all modules of **Generative AI for Commodities Trading**:
- Module 0: Foundations (LLMs, APIs, prompting)
- Module 1: Report Processing (EIA, USDA extraction)
- Module 2: RAG for Research (knowledge bases, retrieval)
- Module 3: Sentiment Analysis (news, social media)
- Module 4: Fundamentals Modeling (supply/demand, integrated models)
- Module 5: Signal Generation (pipeline, backtesting)
- Module 6: Production Systems (deployment, monitoring)

### Next Steps

1. **Deploy Your System**: Take your pipeline to production
2. **Iterate and Improve**: Monitor performance and refine prompts
3. **Expand Coverage**: Add more commodities and data sources
4. **Build Domain Expertise**: Deep dive into specific commodity markets

### Additional Resources

- [Prometheus](https://prometheus.io/) - Metrics and alerting
- [Grafana](https://grafana.com/) - Visualization
- [ELK Stack](https://www.elastic.co/elk-stack) - Log aggregation
- "Site Reliability Engineering" by Google
- "The Art of Monitoring" by James Turnbull

---

**Congratulations on completing the course! You're now equipped to build production GenAI systems for commodity markets.**